# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a walk-through for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the Croissant dataset metadata
dataset = mlc.Dataset(url)

# Access the metadata (dataset.metadata is an object, not dict or list)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Dataset ID (@id): {metadata.id}")
print(f"Date Published: {metadata.date_published}")
print(f"License: {metadata.license}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Review available record sets, their fields, columns, and IDs.

> **Note:** All entities are referenced by their `@id` fields. Record sets, fields, and columns are uniquely identified using these IDs in the Croissant schema.

In [ ]:
# Discover record sets in the Croissant package
record_sets = dataset.record_sets

print('Available record sets and their IDs:')
record_set_ids = []
for rs in record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}")
    record_set_ids.append(rs.id)
    # Display fields/columns for each record set
    print("  Fields and Columns:")
    for fld in rs.fields:
        print(f"    Field: {fld.name}, @id: {fld.id}, Data type: {fld.data_type}")
    for col in rs.columns:
        print(f"    Column: {col.name}, @id: {col.id}, Data type: {col.data_type}")
    print("  ---")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis.

> Use the record set and field `@id`s discovered above to reference specific entities.

In [ ]:
# Extract data from all discovered record sets using their @id
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set with @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set {record_set_id}")
    print(f"Fields/columns in DataFrame: {df.columns.tolist()}")
    print("Preview:")
    display(df.head())
    print("-----")

## 4. Exploratory Data Analysis (EDA)
Perform typical data processing steps: filter records based on criteria, normalize numeric fields, and group data for further analysis. Operations use `@id`s for referencing fields.

- Remove outliers, transform distributions, or group by attributes.

In [ ]:
# Example: Select a numeric field & perform filtering/normalization
# For demonstration, use the first record set and a numeric field/column
first_record_set_id = record_set_ids[0]
df = dataframes[first_record_set_id]

# Identify a numeric field/column by its @id
# Find suitable numeric fields
numeric_field_ids = [fld.id for fld in dataset.record_sets[0].fields if fld.data_type in ['Float', 'Integer', 'Number']]
if not numeric_field_ids:
    numeric_field_ids = [col.id for col in dataset.record_sets[0].columns if col.data_type in ['Float', 'Integer', 'Number']]

if numeric_field_ids:
    numeric_field = numeric_field_ids[0]
    print(f"Using numeric field/column: {numeric_field}")
else:
    numeric_field = df.select_dtypes(include='number').columns[0] if not df.select_dtypes(include='number').empty else df.columns[0]
    print(f"Fallback numeric/field: {numeric_field}")

# Filter: Only keep records where numeric_field > threshold (e.g., 10)
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize numeric_field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a field/column (using @id)
# Try to find a suitable categorical field
group_field_ids = [fld.id for fld in dataset.record_sets[0].fields if fld.data_type == 'Text']
if not group_field_ids:
    group_field_ids = [col.id for col in dataset.record_sets[0].columns if col.data_type == 'Text']

if group_field_ids and group_field_ids[0] in filtered_df.columns:
    group_field = group_field_ids[0]
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships using DataFrames extracted from the record sets.

> Use matplotlib or seaborn for simple visualizations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the filtered numeric field distribution
if numeric_field in filtered_df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(filtered_df[numeric_field], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field} (filtered > {threshold})")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

# Scatter plot between numeric_field and normalized values
if f"{numeric_field}_normalized" in filtered_df.columns:
    plt.figure(figsize=(6,4))
    plt.scatter(filtered_df[numeric_field], filtered_df[f"{numeric_field}_normalized"], c='b')
    plt.xlabel(numeric_field)
    plt.ylabel(f"{numeric_field}_normalized")
    plt.title(f"{numeric_field} vs. normalized values")
    plt.show()

# Grouped visualization (if group_field is defined)
if 'group_field' in locals() and group_field in filtered_df.columns:
    plt.figure(figsize=(8,4))
    sns.barplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore a Croissant FAIR^2 dataset using the `mlcroissant` library:
- Dataset metadata and structure are accessible via the `mlcroissant` API.
- All entities (record sets, fields, columns) are referenced by their `@id`, ensuring consistency.
- Dataframes for each record set enable pandas-based analysis: filtering, normalization, and grouping.
- Basic visualizations reveal distributions and relationships between fields.

Further analysis can be performed based on domain-specific questions or hypotheses using the rich, structured metadata from the Croissant schema.